# 4주차 A5 · UCI HAR 기준 CNN + CBAM (단독 효과)

A0(4주차 기준 CNN)에 **CBAM(채널 + 시간 attention)** 한 가지만 적용한 단독 효과 실험입니다.

> 기준 CNN의 특징에 채널 중요도와 시점 중요도를 차례로 곱하면 더 좋은 모델이 되는가?

| 부분 | A0 | **A5** |
|---|---|---|
| 특징 추출 | Conv(32,k3)+BN → Pool → Conv(64,k3)+BN → Pool | Conv(32,k3)+BN → **CBAM** → Pool → Conv(64,k3)+BN → **CBAM** → Pool |
| 분류 헤드 | GAP → Dropout 0.4 → Dense 6 | 동일 |
| 학습 조건 | Adam 1e-3, batch 64, 최대 100 에폭, EarlyStopping(patience 10) | 동일 |
| 파라미터(손계산) | 7,878 → 약 9,294 (CBAM 두 개 +1,416) | |

따라서 같은 seed에서 `A5 − A0` 차이를 이 기법의 단독 효과로 해석합니다.

**실행 상태:** Colab 실행 전 실험 설계본입니다.

**중요:** 모델 선택은 피험자 단위 validation으로 하고, 공식 test는 최종 모델을 정한 뒤 한 번만 확인합니다.


# 실험 A5: A0 + CBAM (검증셋으로 판단, 테스트는 마지막에 한 번)

| 셀 | 내용 | 언제 실행? |
|---|---|---|
| **0~12** | 학습 + **검증셋** 평가 | SEED 0, 1, 2 로 3번 |
| **13 (🔒)** | **최종 테스트** (잠금: `FINAL_TEST=False`) | 4주차 비교가 끝나고 최종 모델을 정한 뒤 **한 번만** |

> ⚠️ "모두 실행"을 누르면 마지막 셀은 잠금 메시지(AssertionError)를 내고 멈춥니다. 정상 동작입니다.


## 0. 라이브러리와 실험 설정

필요한 패키지를 불러오고 이 실험의 변경점과 실행 방법을 코드 주석에서 확인합니다.



In [ ]:
# ============================================================
# 실험 A5: A0 + CBAM
# ------------------------------------------------------------
# 기반: 4week/05_UCI_HAR_A0_Baseline_CNN.ipynb (데이터·헤드·학습·평가 동일)
#
# 목적: 5가지 CNN 기법 중 CBAM(채널 + 시간 attention) 의 단독 효과 측정
#   CBAM = Channel attention(어떤 채널) → Temporal attention(어느 시점) 을 차례로 곱함
#   A3(SE) 와 같은 위치에 넣어 A3 와도 바로 비교 가능하게 함
#   순서는 원 논문 기본값 Channel → Temporal 로 고정 (순서 비교는 효과 확인 후)
#
# 변경 요약 (Ctrl+F 로 "[A5 변경]" 검색, [A0 변경]·[S0 변경] 은 이전 단계에서 온 변경)
#   1) layers import 추가 (CBAM 블록 정의용)
#   2) cbam_1d 함수 정의 (00_CNN_Blocks_Concept_Guide 와 동일, ratio 8, temporal kernel 7)
#   3) conv1+BN 뒤, conv2+BN 뒤에 CBAM 블록 삽입
#
# 실행 방법: SEED 를 0 → 1 → 2 로 바꿔 3번 실행하고 SUMMARY 기록. 마지막 🔒 셀은 아직 실행하지 않음
# ============================================================

import os
import time   # [A0 변경] 학습·추론 시간 측정
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Dense
from tensorflow.keras.layers import GlobalAveragePooling1D       # [A0 변경] Flatten 대신 전역 평균 풀링
from tensorflow.keras.layers import BatchNormalization, Dropout   # [E1a 변경] 규제용 층
from tensorflow.keras.models import Model
from tensorflow.keras import layers   # [A5 변경] attention 블록 정의용
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping             # [E1a 변경] 조기 종료

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)




## 1. Seed

난수 시드를 고정합니다. 시드를 바꿔 재실행하면 초기화에 따른 변동을 볼 수 있습니다.



In [ ]:
# ============================================================
# 1. Seed
# ============================================================

# [E0 변경] 0, 1, 2 로 바꿔 3번 실행 (한 번의 결과는 우연일 수 있음)
SEED = 0

np.random.seed(SEED)
tf.random.set_seed(SEED)

# [A0 변경] B1·B2 와 같은 방식으로 시드 고정
# 이유: 같은 seed 끼리 A0 와 B1·B2 를 비교하므로 난수 설정도 똑같이 맞춤
#       set_random_seed 는 Python·NumPy·TF 난수를 한 번에 고정하고,
#       enable_op_determinism 은 GPU 연산 순서에 따른 작은 차이까지 줄임
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass




## 2. Data path

Colab에서 Drive를 연결하고 UCI HAR 데이터의 위치를 확인합니다.



In [ ]:
# ============================================================
# 2. Data path
# ============================================================

# [E0 변경] Drive 마운트 + 프로젝트 데이터 경로
from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/가천대학교/2026-2/1. 기계학습프로그래밍/2week/DATA/UCI-HAR Dataset"

assert os.path.exists(DATA_PATH), f"데이터 경로를 확인하세요: {DATA_PATH}"




## 3. Signal names

입력으로 사용할 가속도계·자이로스코프 9개 채널의 파일명을 정의합니다.



In [ ]:
# ============================================================
# 3. Signal names
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]




## 4. Load dataset

공식 train/test 분할의 센서 창, 행동 라벨, 피험자 번호를 읽습니다.



In [ ]:
# ============================================================
# 4. Load dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        data = np.loadtxt(file_path)

        X.append(data)

    # (9, N, 128) -> (N, 128, 9)
    X = np.stack(X, axis=-1)

    y_path = f"{DATA_PATH}/{split}/y_{split}.txt"

    y = np.loadtxt(y_path).astype(int)

    # label 1~6 -> 0~5
    y = y - 1

    # [E0 변경] 피험자 번호(1~30) 로드
    subject_path = f"{DATA_PATH}/{split}/subject_{split}.txt"

    subject = np.loadtxt(subject_path).astype(int)

    return X, y, subject


X_train, y_train, s_train = load_data("train")
X_test, y_test, s_test = load_data("test")




## 4-1. Subject-wise validation split

피험자 5·11·17·23·29를 검증용으로 떼어 학습 피험자와 겹치지 않게 합니다.



In [ ]:
# ============================================================
# 4-1. Subject-wise validation split
# ============================================================

# [E0 변경] 피험자 5명을 통째로 검증셋으로 분리 (같은 사람이 학습·검증에 걸치지 않게)
VAL_SUBJECTS = [5, 11, 17, 23, 29]

val_mask = np.isin(s_train, VAL_SUBJECTS)

X_val, y_val = X_train[val_mask], y_train[val_mask]

# [S0 변경] 검증 윈도우의 피험자 번호 보관
# 이유: 검증 피험자별(특히 5번, 17번) 정확도를 계산하기 위함
s_val = s_train[val_mask]

X_train, y_train = X_train[~val_mask], y_train[~val_mask]


print("\nDataset shape")
print("X_train:", X_train.shape, "(16 subjects)")
print("X_val  :", X_val.shape, "subjects:", VAL_SUBJECTS)
print("X_test :", X_test.shape, "(9 subjects, 마지막 🔒 셀에서만 사용)")




## 5. Normalization

학습 데이터의 평균·표준편차만 사용해 검증·테스트 입력을 변환합니다.



In [ ]:
# ============================================================
# 5. Normalization
# ============================================================

# [E0 변경] 평균/표준편차는 학습 16명으로만 계산
mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True)

X_train = (X_train - mean) / (std + 1e-8)
X_val = (X_val - mean) / (std + 1e-8)
X_test = (X_test - mean) / (std + 1e-8)




## 6. One-hot encoding

정수 행동 라벨을 6개 범주의 원핫 벡터로 바꿉니다.



In [ ]:
# ============================================================
# 6. One-hot encoding
# ============================================================

n_outputs = 6

y_train_onehot = to_categorical(
    y_train,
    num_classes=n_outputs
)

y_val_onehot = to_categorical(
    y_val,
    num_classes=n_outputs
)




## 7. CNN Model (A5: A0 + CBAM)

A3(SE)와 같은 위치, 즉 각 Conv+BN 뒤(MaxPool 전)에 CBAM 블록을 추가합니다. 헤드는 A0와 같습니다.


In [ ]:
# ============================================================
# 7. CNN Model (A5: A0 + CBAM)
# ============================================================

# [A5 변경] 1D CBAM 블록 (00_CNN_Blocks_Concept_Guide 의 코드와 동일, Channel → Temporal 순서)
# 하는 일: 특징맵 (batch, 시간, 채널) 에
#   1) Channel attention : 시간축 평균·최대를 같은 MLP 에 넣어 채널마다 0~1 중요도 (batch, 1, C)
#   2) Temporal attention: 채널축 평균·최대를 Conv1D(kernel 7) 에 넣어 시점마다 0~1 중요도 (batch, T, 1)
# 를 차례로 곱함 → 출력 shape 은 입력과 같음
# SE 와 차이: SE 는 "어떤 채널" 만, CBAM 은 "어떤 채널 + 어느 시점" 까지 조절
# Lambda 를 쓰는 이유: 채널축 평균·최대(tf.reduce_mean/max)를 Keras 층으로 감싸기 위함
def cbam_1d(x, reduction_ratio=8, temporal_kernel=7, name="cbam"):
    channels = int(x.shape[-1])
    hidden = max(channels // reduction_ratio, 1)

    # Channel attention의 두 pooling 경로가 같은 MLP를 공유합니다.
    shared_reduce = layers.Dense(hidden, activation="relu", name=f"{name}_shared_reduce")
    shared_expand = layers.Dense(channels, name=f"{name}_shared_expand")

    avg_descriptor = layers.GlobalAveragePooling1D(name=f"{name}_channel_avg")(x)
    max_descriptor = layers.GlobalMaxPooling1D(name=f"{name}_channel_max")(x)

    avg_score = shared_expand(shared_reduce(avg_descriptor))
    max_score = shared_expand(shared_reduce(max_descriptor))
    channel_score = layers.Add(name=f"{name}_channel_add")([avg_score, max_score])
    channel_score = layers.Activation("sigmoid", name=f"{name}_channel_sigmoid")(channel_score)
    channel_score = layers.Reshape((1, channels), name=f"{name}_channel_reshape")(channel_score)
    x = layers.Multiply(name=f"{name}_channel_scale")([x, channel_score])

    # Temporal attention: 각 시점에서 채널 평균과 최댓값을 사용합니다.
    temporal_avg = layers.Lambda(
        lambda tensor: tf.reduce_mean(tensor, axis=-1, keepdims=True),
        name=f"{name}_temporal_avg",
    )(x)
    temporal_max = layers.Lambda(
        lambda tensor: tf.reduce_max(tensor, axis=-1, keepdims=True),
        name=f"{name}_temporal_max",
    )(x)
    temporal_descriptor = layers.Concatenate(
        axis=-1, name=f"{name}_temporal_concat"
    )([temporal_avg, temporal_max])
    temporal_score = layers.Conv1D(
        1,
        temporal_kernel,
        padding="same",
        activation="sigmoid",
        use_bias=False,
        name=f"{name}_temporal_conv",
    )(temporal_descriptor)

    return layers.Multiply(name=f"{name}_temporal_scale")([x, temporal_score])


inputs = Input(shape=(128, 9))


conv1 = Conv1D(
    filters=32,
    kernel_size=3,
    activation="relu"
)(inputs)

conv1 = BatchNormalization()(conv1)   # [E1a 변경] 학습 안정화

# [A5 변경] CBAM: 채널 중요도 → 시점 중요도 순서로 곱함 (shape 유지: (batch, 126, 32))
conv1 = cbam_1d(conv1, reduction_ratio=8, temporal_kernel=7, name="cbam1")

pool1 = MaxPooling1D(
    pool_size=2
)(conv1)


conv2 = Conv1D(
    filters=64,
    kernel_size=3,
    activation="relu"
)(pool1)

conv2 = BatchNormalization()(conv2)   # [E1a 변경] 학습 안정화

# [A5 변경] CBAM: 64 개 채널 + 61 개 시점 재가중 (shape 유지: (batch, 61, 64))
conv2 = cbam_1d(conv2, reduction_ratio=8, temporal_kernel=7, name="cbam2")

pool2 = MaxPooling1D(
    pool_size=2
)(conv2)


# [A0 변경] Flatten → Dropout → Dense(128) → Dropout  ⇒  GlobalAveragePooling1D → Dropout(0.4)
# 이유: B1·B2 의 헤드와 똑같이 맞춰, 4주차 비교에서 "블록 구조" 차이만 남기기 위함
#       GAP 는 64 개 필터마다 30 개 시점을 평균 내서 64 개 값으로 요약함
#       → Flatten(30×64=1,920 개 값) + Dense(128) 보다 파라미터가 크게 줄어듦
#       Dropout 0.4 와 Dense(128) 제거도 B1·B2 헤드와 같게 하기 위함
# 참고: 2주차 E1b 도 GAP 를 썼지만 Dense(128)·Dropout 0.5 를 유지했으므로 A0 와는 다름
gap = GlobalAveragePooling1D()(pool2)

gap = Dropout(0.4)(gap)


outputs = Dense(
    n_outputs,
    activation="softmax"
)(gap)




## 8. Model

모델을 만들고 손실 함수, 옵티마이저, 평가 지표를 설정합니다.



In [ ]:
# ============================================================
# 8. Model
# ============================================================

model = Model(
    inputs=inputs,
    outputs=outputs
)


model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


model.summary()




## 9. Train

학습 피험자의 데이터로 모델을 훈련하고 학습 이력을 저장합니다.



In [ ]:
# ============================================================
# 9. Train
# ============================================================

# [E1a 변경] val_loss 가 10 에폭 연속 나아지지 않으면 멈추고 최적 가중치로 복원
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

# [A0 변경] 학습 시간 측정 (B1·B2 와 비교)
train_start = time.perf_counter()

history = model.fit(
    X_train,
    y_train_onehot,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_onehot),
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

training_seconds = time.perf_counter() - train_start

print(f"training time: {training_seconds:.1f} sec")


## 10. Validation evaluation   [S0 추가]

검증 피험자에 예측을 적용해 정확도·Macro F1과 혼동 행렬을 계산합니다.



In [ ]:
# ============================================================
# 10. Validation evaluation   [S0 추가]
# ============================================================

# [S0 변경] test 대신 검증셋으로 예측하고 평가
# 이유: 방법 비교·선택은 검증셋으로만 함. test 는 마지막 🔒 셀에서 한 번만
y_val_pred = np.argmax(
    model.predict(X_val, verbose=0),
    axis=1
)

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

val_accuracy = accuracy_score(y_val, y_val_pred)

val_f1 = f1_score(y_val, y_val_pred, average="macro", zero_division=0)

cm_val = confusion_matrix(y_val, y_val_pred, labels=range(n_outputs))


print("\n==============================")
print("VALIDATION RESULTS (subjects 5, 11, 17, 23, 29)")
print("==============================")

print(f"Accuracy  : {val_accuracy:.4f}")
print(f"F1-score  : {val_f1:.4f}")

print(
    classification_report(
        y_val,
        y_val_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

print("CONFUSION MATRIX (validation)")
print(cm_val)




## 11. SIT/STAND focus on validation subjects   [S0 추가]

검증 피험자별 SIT/STAND 정확도를 보고 5·17번 핵심 지표와 나머지 지표를 분리합니다.



In [ ]:
# ============================================================
# 11. SIT/STAND focus on validation subjects   [S0 추가]
# ============================================================

# [S0 변경] 검증 피험자별 전체 정확도 + SIT/STAND 정확도
# SIT/STAND 정확도 = 실제로 앉음·섬인 윈도우 중에서 정확히 맞힌 비율 (6개 클래스 예측 기준)
# 이유: 전체 정확도에는 걷기·눕기가 섞여 있어 SIT/STAND 개선이 묻힘 → 문제 클래스만 따로 측정
SIT, STAND = 3, 4

FOCUS_SUBJECTS = [5, 17]   # 기울기 차이가 작은 검증 피험자 (5번 12.1°, 17번 14.9°)

print("\n==============================")
print("PER-SUBJECT (validation)")
print("==============================")
print("subject | all acc | SIT/STAND acc | SIT->STAND | STAND->SIT")

for s in VAL_SUBJECTS:

    m = (s_val == s)

    ms = m & np.isin(y_val, [SIT, STAND])

    sit_to_stand = np.sum(m & (y_val == SIT) & (y_val_pred == STAND))

    stand_to_sit = np.sum(m & (y_val == STAND) & (y_val_pred == SIT))

    mark = "   <- 핵심" if s in FOCUS_SUBJECTS else ""

    print(
        f"{s:7d} | {accuracy_score(y_val[m], y_val_pred[m]):.4f}  | "
        f"{accuracy_score(y_val[ms], y_val_pred[ms]):.4f}        | "
        f"{sit_to_stand:10d} | {stand_to_sit:10d}{mark}"
    )

# 핵심 지표: 5번 + 17번의 SIT/STAND 윈도우를 합쳐서 계산한 정확도
mf = np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])

focus_ss_acc = accuracy_score(y_val[mf], y_val_pred[mf])

# 보조 지표: 나머지 검증 피험자(11, 23, 29)의 SIT/STAND 정확도
mo = ~np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])

other_ss_acc = accuracy_score(y_val[mo], y_val_pred[mo])

val_ss_err = cm_val[SIT, STAND] + cm_val[STAND, SIT]

val_dynamic_err = cm_val[0:3, 0:3].sum() - np.trace(cm_val[0:3, 0:3])

best_epoch = int(np.argmin(history.history["val_loss"])) + 1


# [A0 변경] 샘플당 추론 시간 측정 (B1·B2 와 같은 방법)
# 처음 몇 번은 그래프 준비 시간이 섞이므로 warm-up 3번 후 20번 반복해 평균
# GPU 종류에 따라 절대값이 달라지므로 같은 Colab 런타임에서 잰 값끼리만 비교
timing_batch = tf.convert_to_tensor(X_val[:256].astype("float32"))

for _ in range(3):
    model(timing_batch, training=False).numpy()

repeats = 20

inference_start = time.perf_counter()

for _ in range(repeats):
    model(timing_batch, training=False).numpy()

inference_seconds = time.perf_counter() - inference_start

inference_ms_per_sample = inference_seconds * 1000 / (repeats * len(timing_batch))


# [A5 변경] SUMMARY 이름 A5 (항목은 A0·B1·B2 와 동일)
print("\n==============================")
print("SUMMARY (결과표에 기록)")
print("==============================")

print(
    f"A5 | seed {SEED} | val acc {val_accuracy:.4f} | val F1 {val_f1:.4f} | "
    f"focus SIT/STAND(5,17) {focus_ss_acc:.4f} | other SIT/STAND(11,23,29) {other_ss_acc:.4f} | "
    f"val SIT<->STAND {val_ss_err} ({cm_val[SIT, STAND]}+{cm_val[STAND, SIT]}) | val dynamic {val_dynamic_err} | "
    f"epochs {len(history.history['val_loss'])} (best {best_epoch}) | params {model.count_params():,} | "
    f"train sec {training_seconds:.1f} | inference ms/sample {inference_ms_per_sample:.4f}"
)




## 12. Learning curve

학습 곡선을 보고 손실과 검증 지표가 에폭에 따라 어떻게 변하는지 살펴봅니다.



In [ ]:
# ============================================================
# 12. Learning curve
# ============================================================

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("epoch")
plt.title(f"A5 loss (seed {SEED})")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train acc")
plt.plot(history.history["val_accuracy"], label="val acc")
plt.xlabel("epoch")
plt.title(f"A5 accuracy (seed {SEED})")
plt.legend()

plt.tight_layout()
plt.show()

## 결과 기록표

SEED 0·1·2 결과를 기록합니다. 같은 seed의 A0 결과를 옆에 함께 적어 차이를 봅니다.

| 실험 | seed | Val Accuracy | Macro F1 | Focus SIT/STAND (5,17) | Other SIT/STAND (11,23,29) | SIT↔STAND | Params | Best epoch | Train sec | ms/sample |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| A0 | 0 |  |  |  |  |  |  |  |  |  |
| A5 | 0 |  |  |  |  |  |  |  |  |  |
| A0 | 1 |  |  |  |  |  |  |  |  |  |
| A5 | 1 |  |  |  |  |  |  |  |  |  |
| A0 | 2 |  |  |  |  |  |  |  |  |  |
| A5 | 2 |  |  |  |  |  |  |  |  |  |

### 해석 기준

- `A5 − A0`이 CBAM 단독 효과이고, `A5 − A3`은 같은 위치에서 **SE 대신 CBAM**(시간 attention 추가)의 효과입니다.
- 정적 자세(SIT/STAND/LAYING)는 윈도우 전체가 비슷해 시간 attention 이득이 작을 것으로 예상합니다. 이득이 있다면 걷기 3종 오류에서 먼저 보일 가능성이 큽니다.
- 파라미터는 SE와 거의 같지만(+28개) 연산 경로가 늘어 학습·추론 시간이 SE보다 길 수 있습니다.
- 세 seed의 평균·표준편차와, 같은 seed끼리 `A5 − A0` 방향이 반복되는지를 함께 봅니다.


## 🔒 최종 테스트 (한 번만 실행)
4주차 후보(A0~A5, B1~B4)를 **검증 지표**로 비교해 최종 모델을 정한 뒤에만 실행합니다.
실행 순서: 위 셀들을 원하는 SEED 로 실행 → 아래 셀에서 `FINAL_TEST = True` 로 바꿔 실행


In [ ]:
# ============================================================
# 13. FINAL TEST   [S0 추가]
# ============================================================

# [S0 변경] 테스트 평가를 별도 셀로 분리하고 잠금 (A0 와 동일)
# 이유: test 점수를 보고 방법을 고르면 test 에 맞춘 선택이 되어 최종 점수의 의미가 사라짐.
#       검증셋으로 최종 모델을 정한 뒤에만 True 로 바꿔 실행 (위에서 학습한 model 을 그대로 평가)
FINAL_TEST = False

assert FINAL_TEST, (
    "테스트셋은 최종 모델을 정한 뒤 한 번만 봅니다. "
    "지금은 위 셀들의 검증 결과로 비교하세요. (최종 평가 때 FINAL_TEST = True)"
)

y_pred = np.argmax(
    model.predict(X_test, verbose=0),
    axis=1
)

accuracy = accuracy_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

cm = confusion_matrix(y_test, y_pred, labels=range(n_outputs))


print("\n==============================")
print("FINAL TEST RESULTS")
print("==============================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"F1-score  : {f1:.4f}")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

print("CONFUSION MATRIX (test)")
print(cm)


print("\n==============================")
print("PER-SUBJECT ACCURACY (test)")
print("==============================")

subject_acc = {}

for s in np.unique(s_test):

    m = (s_test == s)

    subject_acc[s] = accuracy_score(y_test[m], y_pred[m])

    print(f"subject {s:2d}: {subject_acc[s]:.4f}")

worst_subject = min(subject_acc, key=subject_acc.get)

sit_stand_err = cm[SIT, STAND] + cm[STAND, SIT]

dynamic_err = cm[0:3, 0:3].sum() - np.trace(cm[0:3, 0:3])

print(
    f"\nA5 TEST | seed {SEED} | acc {accuracy:.4f} | macro F1 {f1:.4f} | "
    f"SIT<->STAND {sit_stand_err} | dynamic {dynamic_err} | "
    f"worst subject {worst_subject} ({subject_acc[worst_subject]:.4f})"
)